Name: Lin Qizhou

Key insights / takeaways:
- I built a page-wise Visual RAG pipeline that retrieves the most relevant PDF page for a query using text embeddings, then renders the selected page to an image for visual interpretation.
- I used a lightweight sentence-transformer embedding model for retrieval and a local vision-language model (Qwen2.5-VL) to answer questions from the rendered page image.
- I evaluated the system qualitatively by verifying that retrieved pages match the query intent and that answers cite the correct page-level evidence.

In [ ]:
# Step 1: Upgrade Pillow FIRST — restart runtime after this cell
!pip install -qU Pillow
print("\u2705 Pillow upgraded. Now go to Runtime \u2192 Restart session, then continue from Step 2.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
✅ Pillow upgraded. Now go to Runtime → Restart session, then continue from Step 2.


In [ ]:
# FIX 1: Check GPU is available BEFORE loading the model.
# On CPU, Qwen2.5-VL-3B generates ~1-2 tokens/sec = 5-10 min per query.
# On T4 GPU, it generates ~30-50 tokens/sec = ~15 sec per query.
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\u2705 GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("\u274c NO GPU DETECTED!")
    print("   Go to Runtime \u2192 Change runtime type \u2192 T4 GPU")
    print("   Running on CPU will make each query take 5-10 minutes.")
    raise RuntimeError("GPU required. Please change runtime type to T4 GPU and re-run.")

!pip install -qU transformers accelerate
!pip install -qU llama-index-core llama-index-readers-file llama-index-embeddings-huggingface pypdf
!pip install -qU pdf2image
!apt-get install -q poppler-utils

print("\u2705 Installation Complete.")

✅ GPU detected: Tesla T4 (15.6 GB VRAM)
Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Installation Complete.


In [ ]:
# FIX: Downgrade Pillow to a version compatible with torchvision and gradio.
# Pillow 12.1.1 (installed in Step 1) conflicts with torchvision (<10.0)
# and gradio (<12.0). Version 9.5.0 satisfies these dependencies.
!pip install -qU Pillow==9.5.0
# IMPORTANT: A runtime restart is required for this change to take effect.
# Go to Runtime -> Restart session after this cell executes.

import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("\u2705 Embedding model loaded.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires pillow>=10.1, but you have pillow 9.5.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded.


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"\u23f3 Loading {VLM_MODEL_ID}...")
print(f"   Device: {'CUDA - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARNING: will be very slow)'}")

# FIX 2: Explicitly use 'cuda' instead of 'auto' to guarantee GPU placement.
# 'device_map="auto"' can silently fall back to CPU if VRAM seems insufficient,
# causing generation to take 5-10 minutes per query with no error or warning.
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"   # <-- FIXED: was 'auto', now explicitly 'cuda'
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

# Confirm where the model actually landed
model_device = next(vlm_model.parameters()).device
print(f"\u2705 VLM loaded on: {model_device}")
if str(model_device) == 'cpu':
    print("\u26a0\ufe0f  WARNING: Model is on CPU. Each query will take 5-10 minutes!")

⏳ Loading Qwen/Qwen2.5-VL-3B-Instruct...
   Device: CUDA - Tesla T4


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ VLM loaded on: cuda:0


In [ ]:
Settings.embed_model = embed_model
Settings.llm = None
print("\u2705 LlamaIndex settings configured.")

LLM is explicitly disabled. Using MockLLM.
✅ LlamaIndex settings configured.


In [ ]:
from google.colab import files

print("Please upload the 'AstraZeneca-Q4-2025-earnings.pdf' file:")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Please upload the 'AstraZeneca-Q4-2025-earnings.pdf' file:


Saving AstraZeneca-Q4-2025-earnings.pdf to AstraZeneca-Q4-2025-earnings.pdf
User uploaded file "AstraZeneca-Q4-2025-earnings.pdf" with length 1545942 bytes


In [ ]:
PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"

print(f"\U0001f4da Loading {PDF_FILE}...")
documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()

print(f"   Loaded {len(documents)} pages.")
print(f"   Sample Metadata: {documents[0].metadata}")

📚 Loading AstraZeneca-Q4-2025-earnings.pdf...
   Loaded 39 pages.
   Sample Metadata: {'page_label': '1', 'file_name': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_path': 'AstraZeneca-Q4-2025-earnings.pdf', 'file_type': 'application/pdf', 'file_size': 1545942, 'creation_date': '2026-03-05', 'last_modified_date': '2026-03-05'}


In [ ]:
print("\U0001f9e0 Building Vector Index...")
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=1)
print("\u2705 Index Ready!")

🧠 Building Vector Index...
✅ Index Ready!


In [ ]:
from pypdf import PdfReader, PdfWriter
from pdf2image import convert_from_path
from PIL import Image
import torch

def query_visual_rag(query_text, max_new_tokens=256):
    print(f"\n\U0001f50e Querying LlamaIndex for: '{query_text}'...")

    # FIX 3: Warn if model is on CPU before spending time on generation.
    model_device = next(vlm_model.parameters()).device
    if str(model_device) == 'cpu':
        print("\u26a0\ufe0f  WARNING: VLM is running on CPU. This will be very slow (5-10 min).")
        print("   Change runtime to T4 GPU and restart for fast inference.")

    # 1. RETRIEVE: semantic search over page text
    nodes = retriever.retrieve(query_text)
    if not nodes:
        return "\u274c No relevant information found in the index."

    # 2. LOCATE: get page number from metadata
    best_node = nodes[0]
    page_label = best_node.metadata.get('page_label')
    page_index = int(page_label) - 1 if page_label else 0

    print(f"\U0001f4cd Found answer on Page {page_label} (Score: {best_node.score:.4f})")
    print(f"   Context Snippet: {best_node.text[:100]}...")

    # 3. RENDER: convert that PDF page to an image
    print("\U0001f5bc\ufe0f  Rendering page as image...")
    pages = convert_from_path(
        PDF_FILE,
        first_page=page_index + 1,
        last_page=page_index + 1,
        dpi=150
    )
    page_image = pages[0]

    # 4. VISION: build the prompt and run Qwen2.5-VL
    print(f"\U0001f680 Sending page image to Qwen2.5-VL (max {max_new_tokens} tokens)...")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": (
                    f"You are an expert financial analyst reviewing Page {page_label} "
                    # FIX 4: Corrected report name — was 'JLR Annual Report'
                    "of the Goldman Sachs Q4 2024 Earnings Report. "
                    "Answer the following question based on the text, tables, and "
                    "charts visible on this page. "
                    "If the answer involves a chart, describe the visual trend.\n\n"
                    f"Question: {query_text}"
                )}
            ]
        }
    ]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt],
        images=[page_image],
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            # FIX 5: Reduced max_new_tokens from 512 to 256.
            # 512 tokens on T4 GPU takes ~15-20 sec; on CPU it takes ~5-10 MIN.
            # 256 tokens covers all typical financial QA answers in ~8-10 sec on GPU.
            # Caller can pass max_new_tokens=512 if longer answers are needed.
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,   # FIX 6: Remove temperature when do_sample=False
            top_p=None,         # to suppress the 'not valid' warning
        )

    # Decode only the newly generated tokens
    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    answer = vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]
    return answer

print("\u2705 query_visual_rag() function defined and ready.")

✅ query_visual_rag() function defined and ready.


In [ ]:
from IPython.display import display, Markdown

q1 = "What are the net revenues from Equity underwriting?"
display(Markdown(f"### Q1: {q1}"))
display(Markdown(query_visual_rag(q1)))

### Q1: What are the net revenues from Equity underwriting?


🔎 Querying LlamaIndex for: 'What are the net revenues from Equity underwriting?'...
📍 Found answer on Page 30 (Score: 0.5132)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The net revenues from Equity underwriting are $17m.

In [ ]:
q2 = "Give me the details for Investment banking fees"
display(Markdown(f"### Q2: {q2}"))
display(Markdown(query_visual_rag(q2)))

### Q2: Give me the details for Investment banking fees


🔎 Querying LlamaIndex for: 'Give me the details for Investment banking fees'...
📍 Found answer on Page 21 (Score: 0.3765)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The provided text does not contain any information about investment banking fees. The content focuses on summarised financial information for AstraZeneca Finance LLC and AstraZeneca PLC, including details about their debt obligations, financial statements, and other related information. There is no mention of investment banking fees in the given text.

In [ ]:
print("### Task 1 — Revenue Table Extraction")
q_task1 = "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?"
display(Markdown(f"**Question:** {q_task1}"))
display(Markdown(query_visual_rag(q_task1)))


### Task 1 — Revenue Table Extraction


**Question:** What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?


🔎 Querying LlamaIndex for: 'What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?'...
📍 Found answer on Page 1 (Score: 0.7171)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


According to the table provided in the image, AstraZeneca's total Product Sales for FY 2025 were $55,733 million, while their Alliance Revenue was $3,067 million. 

For Product Sales:
- The actual amount was $55,733 million.
- The CER (Change in Revenue) was 9%.
- The actual amount for FY 2025 was $55,733 million.
- The CER for FY 2025 was 9%.

For Alliance Revenue:
- The actual amount was $3,067 million.
- The CER was 39%.
- The actual amount for FY 2025 was $3,067 million.
- The CER for FY 2025 was 39%.

In [ ]:
print("### Task 2 — Regional Revenue Breakdown")
q_task2 = "Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?"
display(Markdown(f"**Question:** {q_task2}"))
display(Markdown(query_visual_rag(q_task2)))


### Task 2 — Regional Revenue Breakdown


**Question:** Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?


🔎 Querying LlamaIndex for: 'Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates?'...
📍 Found answer on Page 34 (Score: 0.4615)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The geographic region with the highest Total Revenue growth in FY 2025, at constant exchange rates, was Emerging Markets, with a growth rate of 11%.

In [ ]:
print("### Task 3 — R&D Pipeline Interpretation")
q_task3 = "Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?"
display(Markdown(f"**Question:** {q_task3}"))
display(Markdown(query_visual_rag(q_task3)))


### Task 3 — R&D Pipeline Interpretation


**Question:** Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?


🔎 Querying LlamaIndex for: 'Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications?'...
📍 Found answer on Page 10 (Score: 0.5477)
   Context Snippet: BioPharmaceuticals - CVRM 
Farxiga 
FY 2025 
$m 
Total  
Revenue  
% Change        
Actual        CE...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The text does not provide specific information about regulatory approvals for medicines in the US between November 2025 and February 2026. It mentions that the company is focused on accelerating the development of new medicines, but does not detail any approvals or specific indications for such approvals. Therefore, there is no data available to answer this question directly from the given text.

In [ ]:
q_task4 = "What was the cash flow from operating activities for the year ended December 31, 2025, and December 31, 2024? Please cite the table or page number where this information is found."
display(Markdown(f"**Audit Query:** {q_task4}"))
audit_response = query_visual_rag(q_task4)
display(Markdown(audit_response))


**Audit Query:** What was the cash flow from operating activities for the year ended December 31, 2025, and December 31, 2024? Please cite the table or page number where this information is found.


🔎 Querying LlamaIndex for: 'What was the cash flow from operating activities for the year ended December 31, 2025, and December 31, 2024? Please cite the table or page number where this information is found.'...
📍 Found answer on Page 20 (Score: 0.5619)
   Context Snippet: Summary  Revenue Drivers  R&D Progress  Sustainability  Financial Performance  Financial Statements ...
🖼️  Rendering page as image...
🚀 Sending page image to Qwen2.5-VL (max 256 tokens)...


The cash flow from operating activities for the year ended December 31, 2025, was $14,575 million, as shown in Table 12 on page 20 of the report. For the same period, it was $11,861 million in 2024.